# Chapter 4: Distributions of Random Variables

**Haydar Kilic — Artificial Intelligence Engineering**

This notebook covers the fundamental probability distributions in statistics with hands-on Python examples:

| Topic | Content |
|------|---------|
| 1. Normal Distribution | Z-score, percentiles, 68-95-99.7 rule |
| 2. Geometric Distribution | Waiting time until first success |
| 3. Binomial Distribution | Probability of k successes in n trials |
| 4. Negative Binomial Distribution | k-th success on the n-th trial |
| 5. Poisson Distribution | Modeling rare events |

---

In [ ]:
# Loading required libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.special import comb, factorial
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Color palette
BLUE   = '#2E86AB'
ORANGE = '#E07A5F'
GREEN  = '#3D9970'
PURPLE = '#6C5B7B'
GRAY   = '#C0C0C0'

print('Libraries loaded successfully ✓')

---
## 1. Normal Distribution

### 1.1 Theory

The normal distribution is a **unimodal, symmetric, bell-shaped** continuous probability distribution. It is defined by two parameters:

$$X \sim N(\mu, \sigma)$$

- $\mu$: Mean (center of the distribution)
- $\sigma$: Standard deviation (spread)

**Probability density function:**

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \, e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

> 📌 **Important:** Many real-world variables are *approximately* normal; none are exactly normal.

In [ ]:
# ── Normal distributions with different parameters ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Left: Standard normal
x = np.linspace(-4, 4, 300)
axes[0].plot(x, stats.norm.pdf(x, 0, 1), color=BLUE, lw=2.5)
axes[0].fill_between(x, stats.norm.pdf(x, 0, 1), alpha=0.15, color=BLUE)
axes[0].set_title('N(μ=0, σ=1) — Standard Normal', fontweight='bold')
axes[0].set_xlabel('x'); axes[0].set_ylabel('Density')
axes[0].axvline(0, color='red', lw=1.5, ls='--', label='μ=0')
axes[0].legend()

# Middle: Different mean
x2 = np.linspace(3, 35, 300)
axes[1].plot(x2, stats.norm.pdf(x2, 19, 4), color=ORANGE, lw=2.5)
axes[1].fill_between(x2, stats.norm.pdf(x2, 19, 4), alpha=0.15, color=ORANGE)
axes[1].set_title('N(μ=19, σ=4)', fontweight='bold')
axes[1].set_xlabel('x'); axes[1].set_ylabel('Density')
axes[1].axvline(19, color='red', lw=1.5, ls='--', label='μ=19')
axes[1].legend()

# Right: Different standard deviations
x3 = np.linspace(-6, 6, 300)
for sigma, color, label in [(0.5, BLUE, 'σ=0.5'), (1, ORANGE, 'σ=1'), (2, GREEN, 'σ=2')]:
    axes[2].plot(x3, stats.norm.pdf(x3, 0, sigma), color=color, lw=2, label=label)
axes[2].set_title('Different σ values (μ=0)', fontweight='bold')
axes[2].set_xlabel('x'); axes[2].set_ylabel('Density')
axes[2].legend()

plt.tight_layout()
plt.suptitle('Normal Distribution: Effect of Parameters', fontsize=14, fontweight='bold', y=1.02)
plt.savefig('normal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ As σ increases the distribution spreads out; changing μ shifts it left/right.')

---
### 1.2 Standardization with Z-Scores

To compare distributions on different scales, we convert each observation to **standard units**:

$$Z = \frac{\text{observation} - \mu}{\sigma}$$

The Z-score indicates how many standard deviations a value lies **above (+)** or **below (−)** the mean.

> 🔑 **Rule:** Observations with $|Z| > 2$ are generally considered **unusual**.

In [ ]:
# ── SAT vs ACT Comparison ────────────────────────────────────────────────────
# SAT: μ=1500, σ=300   →  Pam: 1800
# ACT: μ=21,  σ=5      →  Jim: 24

z_pam = (1800 - 1500) / 300
z_jim = (24   - 21)   / 5

print('=' * 50)
print('SAT vs ACT — Who performed better?')
print('=' * 50)
print(f"Pam's SAT score : 1800  →  Z = (1800-1500)/300 = {z_pam:.2f}")
print(f"Jim's ACT score : 24    →  Z = (24-21)/5       = {z_jim:.2f}")
print()
winner = "Pam" if z_pam > z_jim else "Jim"
print(f'🏆 {winner} has the higher Z-score relative to their peers!')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_info = [('Pam — SAT', 1500, 300, 1800, BLUE,   axes[0]),
             ('Jim — ACT', 21,   5,   24,   ORANGE, axes[1])]

for title, mu, sigma, value, color, ax in plot_info:
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), color=color, lw=2.5)
    ax.fill_between(x, stats.norm.pdf(x, mu, sigma),
                    where=(x <= value), alpha=0.2, color=color)
    ax.axvline(value, color=color, lw=2, ls='--')
    ax.axvline(mu, color='gray', lw=1.5, ls=':')
    ax.set_title(f'{title}\nZ = {(value-mu)/sigma:.2f}', fontweight='bold')
    ax.set_xlabel('Score'); ax.set_ylabel('Density')
    z_val = (value - mu) / sigma
    ax.text(value, ax.get_ylim()[1]*0.5, f' Z={z_val:.1f}', color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('z_score.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 1.3 Percentiles

A **percentile** is the percentage of observations that fall *below* a given value. Graphically, this is the **area to the left** of the curve.

Calculations in Python:
- Area (cumulative probability): `scipy.stats.norm.cdf(x, mu, sigma)`  
- Inverse (find value): `scipy.stats.norm.ppf(p, mu, sigma)`

In [ ]:
# ── Quality Control — Heinz Ketchup ──────────────────────────────────────────
mu_ketchup    = 36
sigma_ketchup = 0.11

print('=' * 55)
print('Heinz Ketchup Quality Control')
print(f'X ~ N(μ={mu_ketchup}, σ={sigma_ketchup})')
print('=' * 55)

# Question 1: P(X < 35.8)
z_low = (35.8 - mu_ketchup) / sigma_ketchup
p_low = stats.norm.cdf(35.8, mu_ketchup, sigma_ketchup)
print(f'\nQuestion 1: P(X < 35.8 oz)')
print(f'  Z = (35.8 - 36) / 0.11 = {z_low:.2f}')
print(f'  P(X < 35.8) = {p_low:.4f}  →  {p_low*100:.2f}%')

# Question 2: P(35.8 < X < 36.2) — passing quality control
z_high   = (36.2 - mu_ketchup) / sigma_ketchup
p_high   = stats.norm.cdf(36.2, mu_ketchup, sigma_ketchup)
p_interval = p_high - p_low
print(f'\nQuestion 2: P(35.8 < X < 36.2) — passing quality control')
print(f'  Z_low = {z_low:.2f},  Z_high = {z_high:.2f}')
print(f'  P = {p_high:.4f} - {p_low:.4f} = {p_interval:.4f}  →  {p_interval*100:.2f}%')
print(f'\n  ✓ Correct answer: {p_interval*100:.2f}%  (consistent with 93.12% from slides)')

# ── Visualization ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x = np.linspace(35.6, 36.4, 400)
y = stats.norm.pdf(x, mu_ketchup, sigma_ketchup)

def plot_distribution(ax, title, fill_color, low=None, high=None):
    ax.plot(x, y, color='black', lw=2)
    if low is not None and high is not None:
        mask = (x >= low) & (x <= high)
    elif low is not None:
        mask = x <= low
    elif high is not None:
        mask = x >= high
    ax.fill_between(x, y, where=mask, alpha=0.4, color=fill_color)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('oz'); ax.set_ylabel('Density')
    ax.axvline(mu_ketchup, color='gray', ls=':', lw=1.5)

plot_distribution(axes[0], 'P(X < 35.8) — Too little ketchup', BLUE,  low=35.8)
plot_distribution(axes[1], 'P(X < 36.2) — Left area',          GREEN, low=36.2)
plot_distribution(axes[2], 'P(35.8 < X < 36.2) — Passes QC',   ORANGE, low=35.8, high=36.2)

for ax in axes:
    ax.set_xticks([35.8, 36.0, 36.2])

plt.tight_layout()
plt.savefig('quality_control.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Finding Cutoff Values — Body Temperature ─────────────────────────────────
mu_temp    = 98.2
sigma_temp = 0.73

print('=' * 55)
print('Body Temperature: Finding Cutoff Values')
print(f'X ~ N(μ={mu_temp}°F, σ={sigma_temp}°F)')
print('=' * 55)

# Bottom 3% cutoff
z_03 = stats.norm.ppf(0.03)
x_03 = z_03 * sigma_temp + mu_temp
print(f'\nCutoff for bottom 3%:')
print(f'  P(Z < z) = 0.03  →  z = {z_03:.3f}')
print(f'  x = z × σ + μ = {z_03:.3f} × {sigma_temp} + {mu_temp} = {x_03:.1f}°F')

# Top 10% cutoff
z_90 = stats.norm.ppf(0.90)
x_90 = z_90 * sigma_temp + mu_temp
print(f'\nCutoff for top 10%:')
print(f'  P(Z < z) = 0.90  →  z = {z_90:.3f}')
print(f'  x = {z_90:.3f} × {sigma_temp} + {mu_temp} = {x_90:.1f}°F')
print(f'  ✓ Correct answer: 99.1°F (option (c) from slides)')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_arr = np.linspace(95.5, 101, 400)
y_arr = stats.norm.pdf(x_arr, mu_temp, sigma_temp)

# Left: bottom 3%
axes[0].plot(x_arr, y_arr, 'k-', lw=2)
axes[0].fill_between(x_arr, y_arr, where=(x_arr <= x_03), color=BLUE, alpha=0.4)
axes[0].axvline(x_03, color=BLUE, lw=2, ls='--')
axes[0].set_title(f'Bottom 3% → {x_03:.1f}°F', fontweight='bold')
axes[0].text(x_03 - 1.2, max(y_arr)*0.3, f'{x_03:.1f}°F', color=BLUE, fontweight='bold')
axes[0].set_xlabel('Temperature (°F)')

# Right: top 10%
axes[1].plot(x_arr, y_arr, 'k-', lw=2)
axes[1].fill_between(x_arr, y_arr, where=(x_arr >= x_90), color=ORANGE, alpha=0.4)
axes[1].axvline(x_90, color=ORANGE, lw=2, ls='--')
axes[1].set_title(f'Top 10% → {x_90:.1f}°F', fontweight='bold')
axes[1].text(x_90 + 0.1, max(y_arr)*0.3, f'{x_90:.1f}°F', color=ORANGE, fontweight='bold')
axes[1].set_xlabel('Temperature (°F)')

plt.tight_layout()
plt.savefig('cutoff_values.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 1.4 The 68 – 95 – 99.7 Rule (Empirical Rule)

Summary for the normal distribution:

| Interval | Approximate Proportion |
|--------|---------------|
| $\mu \pm 1\sigma$ | 68% |
| $\mu \pm 2\sigma$ | 95% |
| $\mu \pm 3\sigma$ | 99.7% |

In [ ]:
# ── 68-95-99.7 Rule Visualization ────────────────────────────────────────────
mu, sigma = 1500, 300  # SAT scores
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 600)
y = stats.norm.pdf(x, mu, sigma)

fig, ax = plt.subplots(figsize=(13, 6))

# Gradient fill
fill_colors = ['#d0e8f5', '#a3cfed', '#6aa6d3']
fill_labels = ['μ±3σ → 99.7%', 'μ±2σ → 95%', 'μ±1σ → 68%']

for k, (color, label) in enumerate(zip(fill_colors, fill_labels), 1):
    band = 4 - k  # 3, 2, 1
    ax.fill_between(x, y,
                    where=((x >= mu - band*sigma) & (x <= mu + band*sigma)),
                    color=color, alpha=0.9, label=label, zorder=k)

ax.plot(x, y, color='navy', lw=2.5, zorder=10)

# Vertical lines and labels
for k in range(-3, 4):
    val = mu + k*sigma
    ax.axvline(val, color='gray', lw=0.8, ls='--', alpha=0.7)
    ax.text(val, -0.00012, f'{val}\n({k:+d}σ)', ha='center', va='top', fontsize=9)

# Double arrows and percentages
def double_arrow(ax, y_pos, xmin, xmax, pct, color):
    ax.annotate('', xy=(xmax, y_pos), xytext=(xmin, y_pos),
                arrowprops=dict(arrowstyle='<->', color=color, lw=2))
    ax.text((xmin+xmax)/2, y_pos + 0.00005, pct, ha='center', color=color,
            fontweight='bold', fontsize=11)

double_arrow(ax, 0.00115, mu-sigma,   mu+sigma,   '68%',   '#1a6ba0')
double_arrow(ax, 0.00075, mu-2*sigma, mu+2*sigma, '95%',   '#155e8f')
double_arrow(ax, 0.00035, mu-3*sigma, mu+3*sigma, '99.7%', '#0e4d77')

ax.set_title('68-95-99.7 Empirical Rule  (SAT: μ=1500, σ=300)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('SAT Score'); ax.set_ylabel('Density')
ax.legend(loc='upper right', fontsize=11)
ax.set_ylim(-0.0003, None)

plt.tight_layout()
plt.savefig('empirical_rule.png', dpi=150, bbox_inches='tight')
plt.show()

# Numerical verification
print('Numerical verification:')
for k in [1, 2, 3]:
    p = stats.norm.cdf(mu+k*sigma, mu, sigma) - stats.norm.cdf(mu-k*sigma, mu, sigma)
    print(f'  μ ± {k}σ  →  [{mu-k*sigma}, {mu+k*sigma}]  →  {p*100:.2f}%')

---
## 2. Geometric Distribution

### 2.1 Theory

The geometric distribution models how many trials are needed **until the first success**.

**Conditions:**
1. Trials are **independent and identically distributed** (i.i.d.)
2. Each trial has exactly two outcomes: success (p) or failure (1-p)

$$P(\text{first success on trial } n) = (1-p)^{n-1} \cdot p$$

| Parameter | Formula |
|-----------|--------|
| Mean | $\mu = \dfrac{1}{p}$ |
| Standard deviation | $\sigma = \sqrt{\dfrac{1-p}{p^2}}$ |

In [ ]:
# ── Milgram Experiment — Geometric Distribution ───────────────────────────────
p_refuse = 0.35  # probability of refusing to administer shock

print('=' * 55)
print('Milgram Experiment — Geometric Distribution')
print(f'p = {p_refuse} (probability of refusing)')
print('=' * 55)

for n in [1, 3, 10]:
    p_n = (1 - p_refuse)**(n-1) * p_refuse
    print(f'P(first success on trial {n}) = {(1-p_refuse)**(n-1):.4f} × {p_refuse} = {p_n:.4f}')

# Expected value and standard deviation
mu_geo  = 1 / p_refuse
std_geo = np.sqrt((1 - p_refuse) / p_refuse**2)
print(f'\nExpected value  : μ = 1/p = 1/{p_refuse} = {mu_geo:.2f} people')
print(f'Standard deviation: σ = {std_geo:.2f} people')

# Dice example (bonus)
p_die = 1/6
p_6th_roll = (1 - p_die)**5 * p_die
print(f'\nBonus example — Rolling a 6 for the first time on the 6th roll:')
print(f'P = (5/6)^5 × (1/6) = {p_6th_roll:.4f}')

# Visualization
n_vals = np.arange(1, 25)
p_vals = stats.geom.pmf(n_vals, p_refuse)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(n_vals, p_vals, color=PURPLE, alpha=0.8, edgecolor='white', linewidth=0.5)
axes[0].axvline(mu_geo, color=ORANGE, lw=2.5, ls='--', label=f'μ={mu_geo:.2f}')
axes[0].set_title(f'Geometric Distribution (p={p_refuse})', fontweight='bold')
axes[0].set_xlabel('Trial number of first success (n)')
axes[0].set_ylabel('Probability')
axes[0].legend()

# Cumulative
c_vals = np.cumsum(p_vals)
axes[1].step(n_vals, c_vals, color=PURPLE, lw=2.5, where='post')
axes[1].axhline(0.95, color=ORANGE, lw=1.5, ls='--', label='95% threshold')
n95 = np.searchsorted(c_vals, 0.95) + 1
axes[1].axvline(n95, color=GREEN, lw=1.5, ls=':', label=f'n={n95}')
axes[1].set_title('Cumulative Distribution', fontweight='bold')
axes[1].set_xlabel('n'); axes[1].set_ylabel('P(X ≤ n)')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('geometric_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n→ The first success occurs by trial {n95} with 95% probability.')

---
## 3. Binomial Distribution

### 3.1 Theory

The binomial distribution gives the probability of obtaining **exactly k successes in n fixed trials**.

**Conditions:** Independent trials, fixed n, two outcomes (success/failure), fixed p.

$$P(\text{k successes in n trials}) = \binom{n}{k} p^k (1-p)^{n-k}$$

| Parameter | Formula |
|-----------|--------|
| Mean | $\mu = np$ |
| Standard deviation | $\sigma = \sqrt{np(1-p)}$ |

**Combination formula:**
$$\binom{n}{k} = \frac{n!}{k!(n-k)!}$$

In [ ]:
# ── Combination Function ──────────────────────────────────────────────────────
from math import comb as combination

print('Combination examples:')
print(f'  C(4,1) = {combination(4,1)}')
print(f'  C(9,2) = {combination(9,2)}')
print(f'  C(5,5) = {combination(5,5)}')
print(f'  C(5,0) = {combination(5,0)}')
print()

# Milgram: exactly 1 out of 4 refuses
n, k, p = 4, 1, 0.35
p_manual = combination(n, k) * p**k * (1-p)**(n-k)
p_scipy  = stats.binom.pmf(k, n, p)
print(f'Exactly 1 out of 4 refuses to administer the shock:')
print(f'  C(4,1) × 0.35^1 × 0.65^3 = {combination(4,1)} × {0.35:.4f} × {0.65**3:.4f}')
print(f'  = {p_manual:.4f}  (scipy: {p_scipy:.4f})')

In [ ]:
# ── Gallup Poll — Obesity ─────────────────────────────────────────────────────
p_obese = 0.262

print('=' * 55)
print('Gallup Poll — Obesity Analysis')
print(f'p = {p_obese}  (proportion of Americans who are obese)')
print('=' * 55)

# Question 1: Exactly 8 out of 10 are obese
n1, k1 = 10, 8
p1 = stats.binom.pmf(k1, n1, p_obese)
print(f'\nP(exactly 8 out of 10 are obese):')
print(f'  C(10,8) × {p_obese}^8 × {1-p_obese:.3f}^2 = {combination(10,8)} × ... = {p1:.5f}')
print(f'  → Very low probability!')

# Question 2: Sample of 100
n2 = 100
mu_binom  = n2 * p_obese
std_binom = np.sqrt(n2 * p_obese * (1 - p_obese))
range_low = mu_binom - 2 * std_binom
range_high= mu_binom + 2 * std_binom

print(f'\nSample of 100:')
print(f'  μ = np = {n2} × {p_obese} = {mu_binom:.1f} people')
print(f'  σ = √(np(1-p)) = {std_binom:.2f}')
print(f'  Usual range: {mu_binom:.1f} ± 2×{std_binom:.2f} = ({range_low:.1f}, {range_high:.1f})')

# Homeschooling
p_home, n_home = 0.13, 1000
mu_home  = n_home * p_home
std_home = np.sqrt(n_home * p_home * (1 - p_home))
print(f'\nHomeschooling (13%, n=1000):')
print(f'  μ={mu_home:.0f}, σ={std_home:.1f}')
print(f'  Usual range: ({mu_home - 2*std_home:.1f}, {mu_home + 2*std_home:.1f})')
print(f'  Is 100 within this range? → {"No — unusual!" if 100 < mu_home - 2*std_home else "Yes — usual"}')

In [ ]:
# ── Binomial → Normal approximation as n increases ───────────────────────────
p_fixed  = 0.10
n_values = [10, 30, 100, 300]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, n in enumerate(n_values):
    k_vals    = np.arange(0, n+1)
    binom_pmf = stats.binom.pmf(k_vals, n, p_fixed)

    # Normal approximation
    mu_n  = n * p_fixed
    std_n = np.sqrt(n * p_fixed * (1 - p_fixed))
    x_n   = np.linspace(0, n, 400)

    axes[idx].bar(k_vals, binom_pmf, color=BLUE, alpha=0.7, label='Binomial PMF',
                  width=0.8, edgecolor='white', lw=0.3)
    axes[idx].plot(x_n, stats.norm.pdf(x_n, mu_n, std_n), color=ORANGE,
                   lw=2.5, label=f'N(μ={mu_n:.0f}, σ={std_n:.1f})')

    # Condition check
    condition = 'np≥10 and n(1-p)≥10' if (n*p_fixed >= 10 and n*(1-p_fixed) >= 10) else 'Condition not met'
    axes[idx].set_title(f'n={n}  ({condition})', fontweight='bold')
    axes[idx].set_xlabel('Number of successes (k)')
    axes[idx].set_ylabel('Probability')
    axes[idx].legend(fontsize=9)
    axes[idx].set_xlim(-1, max(20, n*p_fixed + 4*std_n))

plt.suptitle(f'Binomial → Normal Approximation  (p={p_fixed})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('binom_normal.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ As n increases, the binomial distribution converges to the normal.')

In [ ]:
# ── Normal Approximation to Binomial: Facebook Example ───────────────────────
n_fb, p_fb = 245, 0.25
mu_fb  = n_fb * p_fb
std_fb = np.sqrt(n_fb * p_fb * (1 - p_fb))

print('=' * 55)
print('Facebook Heavy Users Analysis')
print(f'n={n_fb}, p={p_fb}')
print('=' * 55)
print(f'μ = {n_fb} × {p_fb} = {mu_fb:.2f}')
print(f'σ = √({n_fb}×{p_fb}×{1-p_fb}) = {std_fb:.2f}')

# P(X >= 70)
z_70     = (70 - mu_fb) / std_fb
p_norm   = 1 - stats.norm.cdf(70, mu_fb, std_fb)
p_exact  = 1 - stats.binom.cdf(69, n_fb, p_fb)  # exact binomial

print(f'\nP(X ≥ 70):')
print(f'  Z = (70 - {mu_fb:.2f}) / {std_fb:.2f} = {z_70:.2f}')
print(f'  Normal approximation : {p_norm:.4f}  ({p_norm*100:.2f}%)')
print(f'  Exact binomial       : {p_exact:.4f}  ({p_exact*100:.2f}%)')
print(f'  Difference           : {abs(p_norm-p_exact):.5f}  (approximation is very good!)')

# Visualization
k_vals    = np.arange(20, 110)
binom_pmf = stats.binom.pmf(k_vals, n_fb, p_fb)
x_norm    = np.linspace(20, 110, 400)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(k_vals, binom_pmf, width=0.8, color=BLUE, alpha=0.6, label=f'Bin({n_fb},{p_fb})',
       edgecolor='white', lw=0.3)
ax.bar(k_vals[k_vals >= 70], binom_pmf[k_vals >= 70], width=0.8,
       color=ORANGE, alpha=0.8, label=f'P(X≥70) = {p_exact:.3f}')
ax.plot(x_norm, stats.norm.pdf(x_norm, mu_fb, std_fb), color='navy',
        lw=2.5, label=f'N({mu_fb:.1f},{std_fb:.2f})')
ax.axvline(70, color=ORANGE, lw=2, ls='--')
ax.set_title('Facebook: Heavy User Probability (Binomial ≈ Normal)', fontweight='bold')
ax.set_xlabel('Number of heavy users (k)')
ax.set_ylabel('Probability')
ax.legend()

plt.tight_layout()
plt.savefig('facebook_binomial.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Negative Binomial Distribution

### 4.1 Theory

The negative binomial distribution gives the probability of observing the **k-th success on the n-th trial**.

$$P(k\text{-th success on trial }n) = \binom{n-1}{k-1} p^k (1-p)^{n-k}$$

**Binomial vs Negative Binomial:**
| | Binomial | Negative Binomial |
|-|-------|---------------|
| Fixed | n (number of trials) | k (number of successes) |
| Of interest | k successes | trial n |
| Last trial | Any outcome | **Must be a success** |

In [ ]:
# ── Psychology Lab Example — Negative Binomial ───────────────────────────────
p_pair    = 0.10   # probability of finding a matching pair
k_target  = 10    # goal: 10 pairs
n_query   = 30    # 10th success on 30th trial

# Formula: C(n-1, k-1) * p^k * (1-p)^(n-k)
p_negbinom = combination(n_query-1, k_target-1) * p_pair**k_target * (1-p_pair)**(n_query-k_target)
p_scipy_nb = stats.nbinom.pmf(n_query - k_target, k_target, p_pair)

print('=' * 55)
print('Psychology Laboratory — Negative Binomial')
print(f'p={p_pair}, k={k_target}, n={n_query}')
print('=' * 55)
print(f'\nP(10th success on the 30th trial):')
print(f'  C(29,9) × {p_pair}^10 × {1-p_pair:.1f}^20')
print(f'  = {combination(29,9):,} × {p_pair**10:.2e} × {(0.9)**20:.4f}')
print(f'  = {p_negbinom:.5f}')
print(f'  (scipy: {p_scipy_nb:.5f})')

# Visualization of the distribution
n_vals   = np.arange(k_target, 80)
pmf_vals = stats.nbinom.pmf(n_vals - k_target, k_target, p_pair)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(n_vals, pmf_vals, color=GREEN, alpha=0.8, edgecolor='white', lw=0.3,
       label=f'Number of trials for k={k_target} successes')
ax.bar([30], [p_negbinom], color=ORANGE, alpha=1.0, edgecolor='black',
       label=f'n=30: P={p_negbinom:.5f}', width=0.8)

mu_nb = k_target / p_pair
ax.axvline(mu_nb, color='red', lw=2, ls='--', label=f'μ = k/p = {mu_nb:.0f}')
ax.set_title(f'Negative Binomial Distribution  (k={k_target}, p={p_pair})', fontweight='bold')
ax.set_xlabel('Total number of trials (n)')
ax.set_ylabel('Probability')
ax.legend()

plt.tight_layout()
plt.savefig('negative_binomial.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n→ Expected number of trials: μ = k/p = {k_target}/{p_pair} = {mu_nb:.0f}')

---
## 5. Poisson Distribution

### 5.1 Theory

The Poisson distribution is used to model how many times **rare events** occur within a given time interval or area.

**Conditions for use:**
- Events are rare
- Population is large
- Events are independent of each other

$$P(\text{k events}) = \frac{\lambda^k e^{-\lambda}}{k!}$$

- $\lambda$: average number of events per unit time  
- $\mu = \sigma^2 = \lambda$

In [ ]:
# ── Power Outage Example ──────────────────────────────────────────────────────
lambda_weekly = 2  # average 2 outages per week

print('=' * 55)
print('Power Outages — Poisson Distribution')
print(f'λ (weekly) = {lambda_weekly}')
print('=' * 55)

# Question 1: Exactly 1 outage in a week
k1 = 1
p_week_1 = stats.poisson.pmf(k1, lambda_weekly)
print(f'\nP(1 outage in a week):')
print(f'  λ^k × e^(-λ) / k! = {lambda_weekly}^1 × e^(-{lambda_weekly}) / 1!')
print(f'  = {lambda_weekly} × {np.exp(-lambda_weekly):.4f} / 1 = {p_week_1:.4f}')

# Question 2: 3 outages on a specific day
lambda_daily = lambda_weekly / 7
k2 = 3
p_day_3 = stats.poisson.pmf(k2, lambda_daily)
print(f'\nP(3 outages in a day):')
print(f'  λ_day = {lambda_weekly}/7 = {lambda_daily:.4f}')
print(f'  P = {lambda_daily:.4f}^3 × e^(-{lambda_daily:.4f}) / 3! = {p_day_3:.6f}')

# Visualization: different λ values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
k_range = np.arange(0, 15)

for lam, color, label in [(1, BLUE,   'λ=1'), (2, ORANGE, 'λ=2'),
                           (5, GREEN,  'λ=5'), (10, PURPLE, 'λ=10')]:
    axes[0].plot(k_range, stats.poisson.pmf(k_range, lam), 'o-',
                 color=color, lw=2, ms=6, label=label)
axes[0].set_title('Poisson PMF — Different λ values', fontweight='bold')
axes[0].set_xlabel('k (number of events)')
axes[0].set_ylabel('Probability')
axes[0].legend()

# Bar chart for λ=2
pmf2       = stats.poisson.pmf(k_range, lambda_weekly)
bar_colors = [ORANGE if k == 1 else BLUE for k in k_range]
axes[1].bar(k_range, pmf2, color=bar_colors, alpha=0.8, edgecolor='white')
axes[1].set_title(f'Poisson(λ={lambda_weekly}) — Weekly outage count', fontweight='bold')
axes[1].set_xlabel('k (number of outages)')
axes[1].set_ylabel('Probability')
axes[1].text(1, pmf2[1] + 0.005, f'P(k=1)\n={pmf2[1]:.3f}',
             ha='center', color=ORANGE, fontweight='bold')

plt.tight_layout()
plt.savefig('poisson_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Comparative Summary of Distributions

In [ ]:
# ── Compare All Distributions ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# 1. Normal
x = np.linspace(-4, 4, 300)
axes[0].plot(x, stats.norm.pdf(x), color=BLUE, lw=2.5)
axes[0].fill_between(x, stats.norm.pdf(x), alpha=0.15, color=BLUE)
axes[0].set_title('Normal  N(0,1)', fontweight='bold', color=BLUE)
axes[0].set_xlabel('x')

# 2. Geometric
n_geo = np.arange(1, 20)
axes[1].bar(n_geo, stats.geom.pmf(n_geo, 0.35), color=PURPLE, alpha=0.8, edgecolor='white')
axes[1].set_title('Geometric  (p=0.35)', fontweight='bold', color=PURPLE)
axes[1].set_xlabel('Trial of first success')

# 3. Binomial
k_b = np.arange(0, 21)
axes[2].bar(k_b, stats.binom.pmf(k_b, 20, 0.35), color=GREEN, alpha=0.8, edgecolor='white')
axes[2].set_title('Binomial  Bin(20, 0.35)', fontweight='bold', color=GREEN)
axes[2].set_xlabel('k successes')

# 4. Negative Binomial
n_nb = np.arange(3, 30)
axes[3].bar(n_nb, stats.nbinom.pmf(n_nb - 3, 3, 0.35), color=ORANGE, alpha=0.8, edgecolor='white')
axes[3].set_title('Negative Binomial  (k=3, p=0.35)', fontweight='bold', color=ORANGE)
axes[3].set_xlabel('Trial of 3rd success')

# 5. Poisson
k_p = np.arange(0, 15)
axes[4].bar(k_p, stats.poisson.pmf(k_p, 3), color='#E91E63', alpha=0.8, edgecolor='white')
axes[4].set_title('Poisson  (λ=3)', fontweight='bold', color='#E91E63')
axes[4].set_xlabel('k events')

# 6. Summary table
axes[5].axis('off')
table_data = [
    ['Distribution', 'Type',       'Parameter', 'μ',     'σ'],
    ['Normal',       'Continuous', 'μ, σ',      'μ',     'σ'],
    ['Geometric',    'Discrete',   'p',         '1/p',   '√((1-p)/p²)'],
    ['Binomial',     'Discrete',   'n, p',      'np',    '√(np(1-p))'],
    ['Neg. Binomial','Discrete',   'k, p',      'k/p',   '√(k(1-p)/p²)'],
    ['Poisson',      'Discrete',   'λ',         'λ',     '√λ'],
]
table = axes[5].table(cellText=table_data[1:], colLabels=table_data[0],
                       loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2.0)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2E86AB')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#f0f4f8')
axes[5].set_title('Distribution Summary Table', fontweight='bold', pad=20)

plt.suptitle('Fundamental Probability Distributions — Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('distributions_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Exercises

Solve the following exercises using Python.

In [ ]:
# ── Exercise 1: Z-Score ───────────────────────────────────────────────────────
# A student scored 78 on a math exam.
# The class mean is 70 and the standard deviation is 8.
# What is the student's Z-score? What percentile do they fall in?

score, mu_exam, sigma_exam = 78, 70, 8

# WRITE YOUR CODE HERE
# z = ...
# percentile = ...
# print(...)

In [ ]:
# ── Exercise 2: Binomial ─────────────────────────────────────────────────────
# A student randomly guesses on a multiple-choice exam with 4 options per question.
# What is the probability of getting exactly 5 out of 20 correct?
# How many questions are expected to be correct?

n_questions = 20
p_correct   = 1/4

# WRITE YOUR CODE HERE
# p_5 = ...
# expected = ...
# print(...)

In [ ]:
# ── Exercise 3: Poisson ──────────────────────────────────────────────────────
# A call center receives an average of 15 calls per hour.
# What is the probability of receiving exactly 2 calls in any given minute?

lambda_hourly = 15
lambda_minute = lambda_hourly / 60

# WRITE YOUR CODE HERE
# p_2 = ...
# print(...)

In [ ]:
# ── Exercise Answers ─────────────────────────────────────────────────────────
print('ANSWERS')
print('='*50)

# 1
score, mu_exam, sigma_exam = 78, 70, 8
z          = (score - mu_exam) / sigma_exam
percentile = stats.norm.cdf(z) * 100
print(f'1) Z = {z:.2f}, Percentile = {percentile:.1f}th percentile')

# 2
n_questions, p_correct = 20, 1/4
p_5      = stats.binom.pmf(5, n_questions, p_correct)
expected = n_questions * p_correct
print(f'2) P(X=5) = {p_5:.4f}, Expected = {expected:.1f}')

# 3
lambda_m = 15/60
p_2      = stats.poisson.pmf(2, lambda_m)
print(f'3) λ_minute = {lambda_m:.4f}, P(X=2) = {p_2:.4f}')

---
## 📋 Chapter Summary

| Distribution | When to Use | Key Function |
|---------|----------------------|-----------------|
| **Normal** | Continuous data, bell curve | `stats.norm.cdf/ppf` |
| **Geometric** | Number of trials until first success | `stats.geom.pmf` |
| **Binomial** | k successes in n trials (fixed n) | `stats.binom.pmf` |
| **Negative Binomial** | n trials for k-th success (fixed k) | `stats.nbinom.pmf` |
| **Poisson** | Rare events, counts per unit time | `stats.poisson.pmf` |

**Important connections:**
- Binomial → Normal: when $np \geq 10$ and $n(1-p) \geq 10$
- Binomial → Poisson: as $n \to \infty$, $p \to 0$, with $np = \lambda$ held constant
- Geometric is a special case of Negative Binomial ($k=1$)

---
